In [4]:
import mlflow 
import mlflow.sklearn
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import MultinomialNB
from xgboost import XGBClassifier
from sklearn.ensemble import RandomForestClassifier,GradientBoostingClassifier
from sklearn.metrics import accuracy_score,precision_score,recall_score,f1_score
import pandas as pd
import numpy as np
import re
import string   
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer 
import nltk
nltk.download('stopwords')  
nltk.download('wordnet')
import os

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\pc\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\pc\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


In [5]:
df = pd.read_csv('https://raw.githubusercontent.com/shazizisazi/Datasets/a954a8e0efef2124456be0c386f4cb7979b928ad/tweet_emotions.csv').drop(columns=['tweet_id'])
df.head()

,sentiment,content
0,empty,@tiffanylue i know i was listenin to bad habi...
1,sadness,Layin n bed with a headache ughhhh...waitin o...
2,sadness,Funeral ceremony...gloomy friday...
3,enthusiasm,wants to hang out with friends SOON!
4,neutral,@dannycastillo We want to trade with someone w...


In [6]:
import logging

logging.basicConfig(level=logging.INFO)

def lemmatization(text):
    try:
        lemmatizer = WordNetLemmatizer()
        return ' '.join([lemmatizer.lemmatize(word) for word in text.split()])
    except Exception as e:
        logging.warning(f"Lemmatization failed: {e}")
        return text

def remove_stopwords(text):
    try:
        stop_words = set(stopwords.words('english'))
        return ' '.join([word for word in text.split() if word not in stop_words])
    except Exception as e:
        logging.warning(f"Stopword removal failed: {e}")
        return text

def remove_numbers(text):
    try:
        return ''.join([i for i in text if not i.isdigit()])
    except Exception as e:
        logging.warning(f"Number removal failed: {e}")
        return text

def lower_case(text):
    try:
        return ' '.join([word.lower() for word in text.split()])
    except Exception as e:
        logging.warning(f"Lowercase conversion failed: {e}")
        return text

def remove_punctuation(text):
    try:
        return text.translate(str.maketrans('', '', string.punctuation))
    except Exception as e:
        logging.warning(f"Punctuation removal failed: {e}")
        return text

def remove_urls(text):
    try:
        return re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
    except Exception as e:
        logging.warning(f"URL removal failed: {e}")
        return text

def remove_small_sentences(text):
    try:
        return ' '.join([word for word in text.split() if len(word) > 2])
    except Exception as e:
        logging.warning(f"Small word removal failed: {e}")
        return text

In [7]:
def normalize_text(df):
    try:
        if 'content' not in df.columns:
            raise ValueError("Column 'content' not found in dataframe")

        df['content'] = df['content'].astype(str)

        df['content'] = df['content'].apply(lemmatization)
        df['content'] = df['content'].apply(remove_stopwords)
        df['content'] = df['content'].apply(remove_numbers)
        df['content'] = df['content'].apply(lower_case)
        df['content'] = df['content'].apply(remove_punctuation)
        df['content'] = df['content'].apply(remove_urls)
        df['content'] = df['content'].apply(remove_small_sentences)

        logging.info("Text normalization completed successfully")
        return df
    
    except Exception as e:
        logging.error(f"Error in text normalization: {e}")
        raise

In [8]:
normalize_text(df)
df.head()

INFO:root:Text normalization completed successfully


,sentiment,content
0,empty,tiffanylue know listenin bad habit earlier sta...
1,sadness,layin bed headache ughhhhwaitin call
2,sadness,funeral ceremonygloomy friday
3,enthusiasm,want hang friend soon
4,neutral,dannycastillo want trade someone houston ticke...


In [9]:
x=df['sentiment'].isin(['happiness','sadness'])
df=df[x]

In [10]:
df.head()

,sentiment,content
1,sadness,layin bed headache ughhhhwaitin call
2,sadness,funeral ceremonygloomy friday
6,sadness,sleep not thinking old friend want married now...
8,sadness,charviray charlene love miss
9,sadness,kelcouch sorry least friday


In [11]:
df['sentiment']=df['sentiment'].replace({'sadness':0,'happiness':1})
df.sample(5)

C:\Users\pc\AppData\Local\Temp\ipykernel_56740\2069291767.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df['sentiment']=df['sentiment'].replace({'sadness':0,'happiness':1})


,sentiment,content
26304,1,last day college today
30586,1,taylorswift mine too hayley great
8640,1,netmogul sound soooo good right mmmm malasadas...
6717,0,think unfair miss sun place work closed writin...
33439,1,woo nachos and ice cream haha


In [14]:
import dagshub
dagshub.init(repo_owner='shivamtiwari83032-collab', repo_name='mlops-mini-project', mlflow=True)
#mlflow.set_tracking_uri('https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow')

mlflow.set_experiment('bow vs tfidf')

INFO:httpx:HTTP Request: GET https://dagshub.com/api/v1/repos/shivamtiwari83032-collab/mlops-mini-project "HTTP/1.1 200 OK"
INFO:httpx:HTTP Request: GET https://dagshub.com/api/v1/user "HTTP/1.1 200 OK"


Initialized MLflow to track repo "shivamtiwari83032-collab/mlops-mini-project"

INFO:dagshub:Initialized MLflow to track repo "shivamtiwari83032-collab/mlops-mini-project"


Repository shivamtiwari83032-collab/mlops-mini-project initialized!

INFO:dagshub:Repository shivamtiwari83032-collab/mlops-mini-project initialized!


<Experiment: artifact_location='mlflow-artifacts:/2391bcb3b85b4554b2794148d7f9ec4c', creation_time=1776330458926, experiment_id='1', last_update_time=1776330458926, lifecycle_stage='active', name='bow vs tfidf', tags={}, trace_location=None, workspace='default'>

In [24]:
vectorizer={
    'bow':CountVectorizer(),
    'tfidf':TfidfVectorizer()
}

In [23]:
algorithm={
    'logRegression':LogisticRegression(),
    'naiveBayes':MultinomialNB(),
    'xgboost':XGBClassifier(),  
    'randomForest':RandomForestClassifier(),
    'gradientBoosting':GradientBoostingClassifier()
    
}

In [25]:
with mlflow.start_run(run_name='all experiments') as parent_run:
    #loop throgh algo aand feature extraction methods
    for algo_name,model in algorithm.items():
        for vectorizer_name,vec in vectorizer.items():
            with mlflow.start_run(run_name=f'{algo_name} with {vectorizer_name}', nested=True) as child_run:
                #feature extraction
                X=vec.fit_transform(df['content'])
                y=df['sentiment']
                
                #train test split
                X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)

                #log preprocessing parameters
                mlflow.log_param('vectorizer',vectorizer_name)  
                mlflow.log_param('algorithm',algo_name)
                mlflow.log_param('test_size',0.2)
                
                #model training
                model=model
                model.fit(X_train,y_train)

                #log model parameters
                if algo_name=='logRegression':
                    mlflow.log_param('C',model.C)
                elif algo_name=='randomForest':
                    mlflow.log_param('n_estimators',model.n_estimators)
                    mlflow.log_param('max_depth',model.max_depth)
                elif algo_name=='xgboost':
                    mlflow.log_param('n_estimators',model.n_estimators)
                    mlflow.log_param('max_depth',model.max_depth)
                elif algo_name=='gradientBoosting':
                    mlflow.log_param('n_estimators',model.n_estimators)
                    mlflow.log_param('learning_rate',model.learning_rate)
                elif algo_name=='multinomialNB':
                    mlflow.log_param('alpha',model.alpha)
                
                #predictions
                y_pred=model.predict(X_test)
                
                #evaluation
                acc=accuracy_score(y_test,y_pred)
                prec=precision_score(y_test,y_pred)
                rec=recall_score(y_test,y_pred)
                f1=f1_score(y_test,y_pred)
                
                #log metrics and model
                mlflow.log_metric('accuracy',acc)
                mlflow.log_metric('precision',prec)
                mlflow.log_metric('recall',rec)
                mlflow.log_metric('f1_score',f1)
                
                mlflow.sklearn.log_model(model,f'{algo_name}_{vectorizer_name}_model')

                mlflow.log_artifact('exp1_bow_vs_tfidf.ipynb')

                print(f'{algo_name} with {vectorizer_name} - Accuracy: {acc}, Precision: {prec}, Recall: {rec}, F1 Score: {f1}')

2026/04/16 15:16:55 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 15:17:00 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


logRegression with bow - Accuracy: 0.7951807228915663, Precision: 0.7869649805447471, Recall: 0.7970443349753694, F1 Score: 0.7919725893294175
🏃 View run logRegression with bow at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/1/runs/2296674a0691423b90e458534de60f7d
🧪 View experiment at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/1


2026/04/16 15:17:18 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 15:17:27 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


logRegression with tfidf - Accuracy: 0.7966265060240963, Precision: 0.7794533459000943, Recall: 0.8147783251231527, F1 Score: 0.7967244701348748
🏃 View run logRegression with tfidf at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/1/runs/acaa93f2443c4fb2a0eadcd446d605d3
🧪 View experiment at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/1


2026/04/16 15:17:46 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 15:17:55 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


...

randomForest with tfidf - Accuracy: 0.7730120481927711, Precision: 0.772, Recall: 0.7605911330049261, F1 Score: 0.7662531017369727
🏃 View run randomForest with tfidf at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/1/runs/6d477e4ec7b6455e94979d95625b620f
🧪 View experiment at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/1


2026/04/16 15:25:32 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 15:25:37 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


gradientBoosting with bow - Accuracy: 0.7228915662650602, Precision: 0.8160919540229885, Recall: 0.5596059113300492, F1 Score: 0.6639392168322619
🏃 View run gradientBoosting with bow at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/1/runs/65ad3f0cd96043f5b016e59215c20454
🧪 View experiment at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/1


2026/04/16 15:26:31 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/04/16 15:26:35 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


gradientBoosting with tfidf - Accuracy: 0.7146987951807229, Precision: 0.8124076809453471, Recall: 0.541871921182266, F1 Score: 0.6501182033096927
🏃 View run gradientBoosting with tfidf at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/1/runs/8f1ce2b675ad437f9f997119d184beaf
🧪 View experiment at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/1
🏃 View run all experiments at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/1/runs/96b43ea28bf14cf6819a335364000f12
🧪 View experiment at: https://dagshub.com/shivamtiwari83032-collab/mlops-mini-project.mlflow/#/experiments/1
